<a href="https://colab.research.google.com/github/cocyce459-oss/Equalizer314/blob/symeathrea/opencode_colab_tunnel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 OpenCode → Tunnelled WebApp (Google Colab)

Turn a fresh Colab runtime into a **full, password-protected, publicly reachable OpenCode web app** in about 3 minutes.

**What this notebook does**
1. Installs the latest **OpenCode** harness (official binary installer, npm fallback)
2. Stores your model provider API key **on the ephemeral VM only** (never printed, `chmod 600`)
3. Launches `opencode web` headless on `0.0.0.0:4096` with **HTTP basic auth** (auto-generated strong password)
4. Exposes it through a **Cloudflare quick tunnel** (`*.trycloudflare.com`, HTTPS, no account needed) — with `ngrok` as an optional alternative
5. Adds management cells: status / logs / restart / stop

> ⚠️ **Security read this first**
> A tunneled coding agent is effectively **remote shell access to the Colab VM** (it can read/write files and run commands — that is its job).
> - The password is auto-generated with `secrets`, and the tunnel URL is random — treat **both** as secrets.
> - Never commit this notebook with executed outputs containing your key/password.
> - Prefer the optional "Colab proxy" cell (Google-account-gated) when you don't actually need a public URL.
> - This is for **your own** runtime. Don't build anything that proxies untrusted strangers into an agent with shell access.

**Requirements:** Google account · a Colab runtime · (optional) an API key for your model provider — or use any key-less/free models OpenCode supports.

*Runtime note:* Colab free runtimes drop idle sessions (~90 min idle / ~12 h max). This notebook cannot change that; just re-run the launch cell when the runtime recycles.


In [ ]:
#@title 1. Install OpenCode (latest); run once
#@markdown Official binary installer first; npm + Node 22 fallback. Safe to re-run.
import os, subprocess, shutil, sys, json

WORK   = "/content/workspace"   # your project root inside the VM
LOGS   = "/content/oc-logs"     # logs + runtime state
os.makedirs(WORK, exist_ok=True)
os.makedirs(LOGS, exist_ok=True)

def sh(cmd):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, text=True)

def have_oc():
    return shutil.which("opencode") is not None

if not have_oc():
    print("Installing OpenCode via official installer ...")
    r = subprocess.run("curl -fsSL https://opencode.ai/install | bash",
                       shell=True, text=True, capture_output=True)
    print(r.stdout[-1500:]); print(r.stderr[-1500:])
    for p in (os.path.expanduser("~/.opencode/bin"), "/root/.opencode/bin", "/usr/local/bin"):
        if os.path.isfile(os.path.join(p, "opencode")):
            os.environ["PATH"] = p + os.pathsep + os.environ.get("PATH", "")
            break

if not have_oc():
    print("Installer unavailable → falling back to npm ...")
    if not shutil.which("node") or \
       int(subprocess.check_output('node -p "process.versions.node.split(\'.\')[0]"',
                                  shell=True, text=True).strip() or 0) < 20:
        sh("curl -fsSL https://deb.nodesource.com/setup_22.x | bash -")
        sh("apt-get install -y -qq nodejs")
    sh("npm install -g opencode-ai --silent")
    for p in ("/usr/local/bin", "/usr/bin"):
        if os.path.isfile(os.path.join(p, "opencode")):
            os.environ["PATH"] = p + os.pathsep + os.environ.get("PATH", "")
            break

assert have_oc(), "OpenCode install failed — check network output above."
print("── version ──")
sh("opencode --version")
print(f"\n✅ OpenCode ready · workspace = {WORK} · logs = {LOGS}")

In [ ]:
#@title 2. Launch OpenCode web server (unprotected)
#@markdown Starts `opencode web` headless, waits for health, and prints access info.
import base64, json, os, secrets, shutil, string, subprocess, time, urllib.request, urllib.error

WORK, PORT = "/content/workspace", 4096

env = os.environ.copy()

log_path = "/content/oc-logs/opencode.log"
proc = subprocess.Popen(
    ["opencode", "web", "--port", str(PORT), "--hostname", "0.0.0.0"],
    cwd=WORK, env=env, stdout=open(log_path, "ab"), stderr=subprocess.STDOUT,
    start_new_session=True,  # detach so it survives this cell
)
json.dump({"user": "", "pass": "", "port": PORT, "pid": proc.pid},
          open("/content/oc-logs/state.json", "w"))

def probe():
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/")
    try:
        return urllib.request.urlopen(req, timeout=4).status
    except urllib.error.HTTPError as e:
        return e.code
    except Exception:
        return 0

ok = False
for _ in range(45):
    time.sleep(2)
    if probe() == 200:
        ok = True
        break

if not ok:
    print("⚠️  server not healthy yet — last log lines:")
    print(open(log_path, errors="replace").read()[-3000:])
else:
    from IPython.display import HTML, display
    display(HTML(
        f"<div style='font-family:monospace;background:#0b1021;color:#d7e3ff;"
        f"padding:14px;border-radius:10px'>"
        f"✅ <b>OpenCode web server is up</b> on port {PORT} (PID {proc.pid})<br>"
        f"local url &nbsp;→ <code>http://127.0.0.1:{PORT}</code></div>"))

In [ ]:
#@title 3. Public HTTPS tunnel; Cloudflare or Ngrok
#@markdown Choose your preferred tunnel provider. If using Ngrok, make sure to supply your authtoken. Press this link for the token okay https://dashboard.ngrok.com/get-started/your-authtoken.
TUNNEL_PROVIDER = "cloudflared" #@param ["cloudflared", "ngrok"]
NGROK_AUTHTOKEN = "" #@param {type:"string"}

import json, os, re, shutil, subprocess, time
from IPython.display import HTML, display

PORT = 4096
state = json.load(open("/content/oc-logs/state.json"))
WORK = "/content/workspace"
os.makedirs(WORK, exist_ok=True)

URL = None

if TUNNEL_PROVIDER == "cloudflared":
    if not shutil.which("cloudflared"):
        print("Downloading cloudflared ...")
        r = subprocess.run(
            "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/"
            "cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
            shell=True, text=True, capture_output=True)
        print(r.stdout[-500:], r.stderr[-500:])
    assert shutil.which("cloudflared"), "cloudflared install failed"

    tlog = "/content/oc-logs/cloudflared.log"
    # Clean old logs
    if os.path.exists(tlog): os.remove(tlog)

    subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=open(tlog, "ab"), stderr=subprocess.STDOUT, start_new_session=True)

    for _ in range(30):
        time.sleep(2)
        if os.path.exists(tlog):
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(tlog, errors="replace").read())
            if m:
                URL = m.group(0)
                break
    assert URL, "Cloudflare tunnel URL not found:\n" + open(tlog, errors="replace").read()[-2000:]

elif TUNNEL_PROVIDER == "ngrok":
    if not NGROK_AUTHTOKEN.strip():
        raise ValueError("Please provide a valid NGROK_AUTHTOKEN to use Ngrok.")

    if not shutil.which("ngrok"):
        print("Downloading ngrok...")
        subprocess.run(
            "curl -fSL -o /tmp/ngrok.tgz https://bin.equinox.io/c/b34t2HcxS4e/ngrok-v3-stable-linux-amd64.tgz && "
            "tar -xzf /tmp/ngrok.tgz -C /usr/local/bin && chmod +x /usr/local/bin/ngrok",
            shell=True, check=True)
    assert shutil.which("ngrok"), "ngrok install failed"

    # Configure token
    subprocess.run(["ngrok", "config", "add-authtoken", NGROK_AUTHTOKEN.strip()], check=True)

    tlog = "/content/oc-logs/ngrok.log"
    if os.path.exists(tlog): os.remove(tlog)

    subprocess.Popen(
        ["ngrok", "http", str(PORT), "--log", "stdout"],
        stdout=open(tlog, "ab"), stderr=subprocess.STDOUT, start_new_session=True)

    for _ in range(30):
        time.sleep(2)
        if os.path.exists(tlog):
            m = re.search(r"url=(https://[-a-z0-9\.]+\.ngrok-free\.app)", open(tlog, errors="replace").read())
            if m:
                URL = m.group(1)
                break
    assert URL, "Ngrok tunnel URL not found:\n" + open(tlog, errors="replace").read()[-2000:]

display(HTML(
    f"<div style='font-family:monospace;background:#10231a;color:#d9ffe9;"
    f"padding:16px;border-radius:10px;font-size:15px'>"
    f"🚀 <b>Your tunneled webapp is live ({TUNNEL_PROVIDER})</b><br><br>"
    f"url &nbsp;&nbsp;&nbsp;&nbsp;→ <a href='{URL}' target='_blank' style='color:#7dffa8'>{URL}</a><br>"
    f"username → <code>{state.get('user', 'None')}</code><br>"
    f"password → <code>{state.get('pass', 'None')}</code><br><br>"
    f"⚠️ anyone with <i>both</i> URL and password can use this agent (shell included). "
    f"Don't post them publicly.</div>"))

In [ ]:
#@title (Optional) 4. Skip the public tunnel; open via Colab's Google-auth proxy
#@markdown Use this when you only need access from your own logged-in browser. Opens the port through Colab's own proxy (gated by your Google account).
from google.colab import output
output.serve_kernel_port_as_iframe(4096)
print("Opened as an iframe in the cell output below (or above).")

In [ ]:
import google.colab.output
help(google.colab.output.serve_kernel_port_as_window)

In [ ]:
#@title 5. Management; status / logs / restart / stop
ACTION = "logs"  #@param ["status", "logs", "restart", "stop"]

import json, os, signal, subprocess, time, base64, urllib.request, urllib.error

WORK, PORT = "/content/workspace", 4096
state_path = "/content/oc-logs/state.json"
log_path = "/content/oc-logs/opencode.log"

def alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

state = json.load(open(state_path)) if os.path.exists(state_path) else {}

if ACTION == "status":
    pid = state.get("pid", -1)
    print("server pid:", pid, "→", "running ✅" if alive(pid) else "not running ❌")
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/")
        tok = base64.b64encode(f"{state['user']}:{state['pass']}".encode()).decode()
        req.add_header("Authorization", f"Basic {tok}")
        print("HTTP health:", urllib.request.urlopen(req, timeout=4).status)
    except Exception as e:
        print("HTTP health: unreachable →", e)

elif ACTION == "logs":
    print(open(log_path, errors="replace").read()[-5000:])

elif ACTION == "stop":
    if state.get("pid") and alive(state["pid"]):
        os.kill(state["pid"], signal.SIGTERM)
        print("stopped PID", state["pid"])
    else:
        print("nothing to stop")

elif ACTION == "restart":
    if state.get("pid") and alive(state["pid"]):
        os.kill(state["pid"], signal.SIGTERM)
        time.sleep(2)
    print("↻ re-run cell 4 (Launch) and cell 5 (Tunnel) to bring it back with fresh credentials.")

In [ ]:
#@title Base64 Tunnel Streamer; Direct Download alternative
#@markdown ---
#@markdown Configure your target file path and server settings below:
FILE_PATH = '/content/oc-logs/cloudflared.log' #@param {type:"string"}
PORT = 2791 #@param {type:"integer"}
#@markdown ---

import os
import sys
import time
import signal
import socket
import subprocess
from google.colab import output
from IPython.display import HTML, display

def safe_cleanup_port(port):
    """
    Terminates any external process listening on the port.
    Returns False if the current Colab kernel itself holds the port.
    """
    current_pid = os.getpid()
    try:
        # Check processes on the port without killing immediately
        res = subprocess.run(["fuser", f"{port}/tcp"], capture_output=True, text=True)
        raw_output = f"{res.stdout} {res.stderr}".replace(f"{port}/tcp:", "")
        pids = [int(p) for p in raw_output.split() if p.isdigit()]

        for pid in pids:
            if pid == current_pid:
                # Port is locked by the Colab kernel from an older thread
                return False
            try:
                os.kill(pid, signal.SIGKILL)
                print(f"✅ Terminated old server process (PID {pid}) on Port {port}.")
            except ProcessLookupError:
                pass
        return True
    except Exception as e:
        print(f"☑️ Port check skipped: {e}")
        return True

def get_free_port():
    """Finds an open ephemeral port dynamically."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

# --- Main Execution ---
if not os.path.exists(FILE_PATH):
    print(f"❌ Error: Configured file not found at '{FILE_PATH}'")
else:
    # 1. Clean up port safely without killing the Colab kernel
    if not safe_cleanup_port(PORT):
        # If the current kernel holds the port from a previous run, don't crash; switch ports!
        NEW_PORT = get_free_port()
        print(f"⚠️ Port {PORT} is held by this kernel session. Auto-switching to Port {NEW_PORT} to prevent crash.")
        PORT = NEW_PORT

    directory = os.path.dirname(os.path.abspath(FILE_PATH))
    file_name = os.path.basename(FILE_PATH)

    # 2. Launch http.server in an isolated background subprocess (NOT a kernel thread)
    server_cmd = [sys.executable, "-m", "http.server", str(PORT), "--directory", directory]
    subprocess.Popen(server_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(0.5)  # Allow socket to bind

    # 3. Render UI with direct proxy stream link
    html_code = f'''
        <div style="padding: 20px; border: 2px solid #1a73e8; border-radius: 12px; background-color: #f8f9fa; max-width: 450px; font-family: 'Google Sans', sans-serif; box-shadow: 0 4px 12px rgba(0,0,0,0.08);">
            <h3 style="margin-top: 0; color: #1a73e8; font-size: 18px;">Direct Download Ready</h3>
            <p style="color: #5f6368; font-size: 14px; line-height: 1.5; margin-bottom: 15px;">The high-speed proxy tunnel is now active. Click below to stream the target payload directly from your Google Colab instance.</p>

            <a id="download-link" href="#" target="_blank" style="text-decoration: none; pointer-events: none; opacity: 0.5;">
                <button id="dl-btn" style="
                    background-color: #1a73e8;
                    color: white;
                    padding: 14px 24px;
                    border: none;
                    border-radius: 6px;
                    cursor: not-allowed;
                    font-size: 15px;
                    font-weight: 600;
                    width: 100%;
                    transition: all 0.2s;
                ">
                    ⏳ Initializing Stream Link...
                </button>
            </a>
            <p id="status" style="font-size: 11.5px; color: #d93025; margin-top: 12px; font-weight: 500; font-family: monospace;">
                Status: Waiting for Google Cloud Proxy...
            </p>
        </div>

        <script>
            (async function() {{
                try {{
                    let baseUrl = await google.colab.kernel.proxyPort({PORT});

                    if (!baseUrl.endsWith('/')) {{
                        baseUrl = baseUrl + '/';
                    }}

                    const fullUrl = baseUrl + encodeURIComponent("{file_name}");

                    const link = document.getElementById('download-link');
                    const btn = document.getElementById('dl-btn');
                    const status = document.getElementById('status');

                    link.href = fullUrl;
                    link.style.pointerEvents = 'auto';
                    link.style.opacity = '1';

                    btn.style.cursor = 'pointer';
                    btn.style.backgroundColor = '#188038';
                    btn.innerText = "🚀 Stream {file_name}";

                    status.style.color = "#188038";
                    status.innerHTML = "<b>Status:</b> Tunnel online (Port {PORT}). Secured high-speed link active.";

                }} catch (e) {{
                    document.getElementById('status').innerText = "Error resolving proxy link: " + e;
                }}
            }})();
        </script>
    '''
    display(HTML(html_code))

In [ ]:
#@title Keep Alive — Infinite Silent Audio {display-mode: "form"}
#@markdown Toggle to run or deactivate a silent audio loop in the background to prevent Colab from disconnecting.
Active = True #@param {type:"boolean"}

import time
from IPython.display import display, Audio, HTML
import numpy as np

if Active:
    print("🚀 Keep-alive script started. Running in background...")
    # Generate 1 second of silence (frequency 0)
    sr = 22050
    silent_wave = np.zeros(sr)

    try:
        while True:
            # Displays an invisible, silent audio player that auto-plays
            display(Audio(silent_wave, rate=sr, autoplay=True))
            time.sleep(60)
    except KeyboardInterrupt:
        print("\n🛑 Keep-alive stopped.")
else:
    display(HTML('''
    <p style="color: gray; font-size: 11px; margin-top: 4px;">⏸️ Infinite silent audio is deactivated.</p>
    '''))

## ✅ Security checklist & ops notes

- [ ] Password saved somewhere safe (cell 4 output / `state.json` on the VM)
- [ ] API key **not** present in any notebook cell or committed output
- [ ] You understand: URL + password = full agent access (read/write/execute) on this VM
- [ ] When done: run `ACTION = "stop"` (cell 7), then `Runtime → Disconnect and delete runtime` — that wipes the key and the agent

**Runtime limits (Colab, not configurable here):** idle sessions disconnect (~90 min) and hard cap (~12 h). After a recycle, just re-run cells 4–5; your `WORK` dir (`/content/workspace`) is gone unless you mount Google Drive first (`from google.colab import drive; drive.mount('/content/drive')` and point `WORK` at a Drive folder).

**Bring-your-own-harness:** the tunnel recipe is harness-agnostic — swap the launch line in cell 4 for any other web UI (e.g. `aider --port ...`, `open-webui serve`) and the same cloudflared cell still works. Keep the basic-auth step if your harness supports it; otherwise put a gate in front before going public.

**Troubleshooting**
| Symptom | Fix |
|---|---|
| `opencode: command not found` | Re-run cell 1; check the installer output block |
| Health probe times out | `ACTION = "logs"` (cell 7) — usually a provider/auth error at boot |
| Tunnel cell finds no URL | Check `/content/oc-logs/cloudflared.log`; re-run cell 5 |
| 401 loop in browser | Re-run cell 4 (password rotates) and log in with the **new** password |
| Need free/key-less models | `opencode models` lists what's reachable; or `opencode auth login` interactively |
